# Cleanup failure after successful work

The body succeeds, but its finalizer fails. A successful body does not make the entire call successful.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [1]:
import asyncio
from aoa.action_machine.graph.core.exclude_graph_model import exclude_graph_model
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.resources.connection_scope import connection_scope
from aoa.action_machine.resources.per_call_connection import PerCallConnection

## 2. Define the objects used in this case

These definitions give the case an observable result without introducing unrelated business behavior.

In [2]:
@exclude_graph_model
class TraceResource(BaseResource):
    """Record lifecycle calls without opening an external connection."""
    def __init__(self, events):
        """Keep the event list supplied by the example."""
        self.events = events
    def get_wrapper_class(self):
        """Allow direct use by nested actions."""
        return None
    async def open(self):
        """Record acquisition."""
        self.events.append("open")
    async def close(self):
        """Record release."""
        self.events.append("close")


def fail_close(resource):
    """Simulate an operational release failure."""
    resource.events.append("close failed")
    raise OSError("release unavailable")

## 3. Prepare this call

Create the state and declarations used below. Merely declaring a per-call factory does not run it.

In [3]:
events = []

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

In [4]:
try:
    async with connection_scope({"db": PerCallConnection(lambda: TraceResource(events), fail_close)}):
        events.append("use")
except OSError as error:
    print(type(error).__name__ + ": " + str(error))
assert events == ["open", "use", "close failed"]
print(events)

OSError: release unavailable
['open', 'use', 'close failed']


## 5. Explain the result

With no earlier error to preserve, the closing error propagates out of the scope. This example reports an attempted release, not a successful release.